# Multimodal Story Prediction — Experiments
**Module:** 55-710365 Deep Neural Networks and Learning Systems  
**Student:** Ashkan Bazrgary  

Each experiment builds on the previous best model. Run one section at a time.

In [ ]:
# ── Setup: clone repo and install dependencies ──────────────────────────────
!pip install -q datasets transformers beautifulsoup4
!git clone https://github.com/AshkanBgz/dnnls_final_project_2.git
import sys
sys.path.insert(0, '/content/dnnls_final_project_2')

In [ ]:
# ── Mount Google Drive (to save checkpoints and results) ────────────────────
from google.colab import drive
drive.mount('/content/gdrive')

import os
DRIVE_DIR = '/content/gdrive/MyDrive/DL_Checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Drive ready:', DRIVE_DIR)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import gc

from torch.utils.data import DataLoader, random_split
from datasets import load_dataset
from transformers import BertTokenizer

from src.model import (
    EncoderLSTM, DecoderLSTM, Seq2SeqLSTM,
    VisualAutoencoder,
    SequencePredictor,
    CrossModalSequencePredictor,
)
from src.utils import SequencePredictionDataset, init_weights, validation
from src.train import run_training

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ── Shared config ────────────────────────────────────────────────────────────
EMB_DIM    = 16
LATENT_DIM = 16
N_EPOCHS   = 25
BATCH_SIZE = 8
LR         = 0.001

COT_CONFIG = dict(
    lambda_reid=0.10,
    lambda_ground_mse=0.10,
    lambda_contrast=0.10,
    lambda_entity_pool=0.05,
    contrastive_tau=0.07,
    use_frame_aware_grounding=True,
    use_contrastive_roi=True,
    use_entity_pooling=True,
)

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
tokenizer = BertTokenizer.from_pretrained('google-bert/bert-base-uncased')

train_raw  = load_dataset('daniel3303/StoryReasoning', split='train')
test_raw   = load_dataset('daniel3303/StoryReasoning', split='test')

sp_train = SequencePredictionDataset(train_raw, tokenizer)
sp_test  = SequencePredictionDataset(test_raw,  tokenizer)

train_size = int(0.8 * len(sp_train))
val_size   = len(sp_train) - train_size
train_subset, val_subset = random_split(sp_train, [train_size, val_size])

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_subset,   batch_size=4,          shuffle=True)
test_loader  = DataLoader(sp_test,      batch_size=4,          shuffle=False)

print(f'Train: {len(train_subset)}  Val: {len(val_subset)}  Test: {len(sp_test)}')

In [ ]:
# ── Load pretrained text autoencoder (frozen) ────────────────────────────────
def load_text_autoencoder(path, device):
    enc = EncoderLSTM(tokenizer.vocab_size, EMB_DIM, EMB_DIM).to(device)
    dec = DecoderLSTM(tokenizer.vocab_size, EMB_DIM, EMB_DIM).to(device)
    model = Seq2SeqLSTM(enc, dec).to(device)
    ckpt = torch.load(path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt)
    model.load_state_dict(state)
    for p in model.parameters():
        p.requires_grad = False
    return model

TEXT_AE_PATH = '/content/dnnls_final_project_2/text_autoencoder.pth'
text_autoencoder = load_text_autoencoder(TEXT_AE_PATH, device)
print('Text autoencoder loaded and frozen.')

In [ ]:
# ── Helper: save loss curve ───────────────────────────────────────────────────
def save_loss_curve(losses, exp_name):
    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(losses)+1), losses, marker='o', label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'{exp_name} — Training Loss over {len(losses)} Epochs')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    fname = exp_name.lower().replace(' ', '_') + '_loss.png'
    plt.savefig(f'{DRIVE_DIR}/{fname}', dpi=150)
    plt.show()
    print(f'Saved: {DRIVE_DIR}/{fname}')
    return fname

---
## Experiment 1 — Cross-modal Attention Fusion
**Change:** Replace simple concat fusion with bidirectional cross-modal attention.  
**Everything else:** CNN encoder, latent dim 16, GRU hidden 16, text decoder frozen.  
**Hypothesis:** Letting image and text embeddings attend to each other before fusion will produce richer representations and lower loss than simple concatenation.

In [ ]:
gc.collect()
torch.cuda.empty_cache()

# Build model
visual_ae_exp1 = VisualAutoencoder(latent_dim=LATENT_DIM).to(device)
visual_ae_exp1.apply(init_weights)

exp1_model = CrossModalSequencePredictor(
    visual_ae_exp1, text_autoencoder, LATENT_DIM, LATENT_DIM
).to(device)

total = sum(p.numel() for p in exp1_model.parameters() if p.requires_grad)
print(f'Exp 1 trainable parameters: {total:,}')

In [ ]:
# Train
val_fn = lambda m, dl: validation(m, dl, tokenizer, device)

exp1_losses = run_training(
    exp1_model, train_loader, val_loader,
    tokenizer, device,
    n_epochs=N_EPOCHS, lr=LR,
    cot_config=COT_CONFIG,
    val_fn=val_fn,
)

print(f'\nExp 1 final loss: {exp1_losses[-1]:.4f}')

In [ ]:
# Save results
save_loss_curve(exp1_losses, 'Experiment 1 Cross-modal Attention')
torch.save(exp1_model.state_dict(), f'{DRIVE_DIR}/exp1_model.pth')
print('Exp 1 model saved to Drive.')